In [16]:
import os
import pandas as pd
import numpy as np

###########   MODALITY READING  ###################################
import warnings; warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
# Root folder containing all datasets
storage_root = r"../Storage Repository"

rows = []

# Loop through datasets (e.g., ACHRP, COBRE, MCIC)
for dataset in os.listdir(storage_root):
    dataset_path = os.path.join(storage_root, dataset)
    data_path = os.path.join(dataset_path, "data")

    # Skip if no data/ folder
    if not os.path.isdir(data_path):
        continue

    # Loop through subjects (sub-*)
    for sub in os.listdir(data_path):
        sub_path = os.path.join(data_path, sub)
        if not (os.path.isdir(sub_path) and sub.startswith("sub-")):
            continue

        # Check sessions (ses-*)
        session_folders = [s for s in os.listdir(sub_path) if os.path.isdir(os.path.join(sub_path, s)) and s.startswith("ses-")]

        # If no sessions, still check directly under subject
        if not session_folders:
            session_folders = [None]

        for ses in session_folders:
            ses_path = os.path.join(sub_path, ses) if ses else sub_path

            # Modality folders
            anat_path = os.path.join(ses_path, "anat")
            func_path = os.path.join(ses_path, "func")
            dwi_path  = os.path.join(ses_path, "dwi")

            # Existence flags (1 if folder exists and has ≥1 file)
            def has_files(path):
                return os.path.isdir(path) and any(os.path.isfile(os.path.join(path, f)) for f in os.listdir(path))

            anat_flag = int(has_files(anat_path))
            func_flag = int(has_files(func_path))
            dwi_flag  = int(has_files(dwi_path))

            session_name = ses.replace("ses-", "") if ses else "NA"

            rows.append({
                "dataset": dataset,
                "participant_id": sub.replace("sub-", ""),
                "session": session_name,
                "anat": anat_flag,
                "fmri": func_flag,
                "dwi": dwi_flag,
                "directory": os.path.abspath(ses_path)  
            })

# Build DataFrame and save combined CSV
df_modalities = pd.DataFrame(rows)
df_modalities.to_csv("modalities_info.csv", index=False)

print(f"Saved {len(df_modalities)} rows to modalities_info.csv")


###########   PARTICIPANT READING  ################################

all_dfs = []

# Loop through datasets (e.g., ACHRP, COBRE, MCIC)
for dataset in os.listdir(storage_root):
    dataset_path = os.path.join(storage_root, dataset)
    participants_file = os.path.join(dataset_path, "participants.csv")

    if os.path.isfile(participants_file):
        try:
            df = pd.read_csv(participants_file)

            # Remove unnamed columns
            df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

            # Add dataset name
            df["dataset"] = dataset

            all_dfs.append(df)

        except Exception as e:
            print(f"❌ Error reading {participants_file}: {e}")

# Concatenate all DataFrames
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)

    combined_df.to_csv("participants_info.csv", index=False)

    print(
        f"\nSaved combined CSV with {len(combined_df)} rows "
        f"across {combined_df['dataset'].nunique()} datasets "
        f"→ participants_info.csv"
    )
else:
    print("⚠️ No participants.csv files found.")




###################################################################




def merge_clinical_data1(df_harmonization, df_source, dataset_name, column_mapping):
    """
    Merge clinical variables (totals and individual items) from df_source into df_harmonization.

    Parameters:
    - df_harmonization: main dataframe (participants + modalities)
    - df_source: dataframe with clinical scores to merge
    - dataset_name: name of the dataset (string)
    - column_mapping: dict mapping source columns -> harmonized columns (e.g., items and totals)
    """

    # Rename source columns to harmonized names
    df_source = df_source.rename(columns=column_mapping)

    # Determine which columns are present in df_source
    present_cols = [col for col in df_source.columns if col in column_mapping.values()]

    # Loop over source rows and update harmonization dataframe
    for idx, row in df_source.iterrows():
        mask = (
            (df_harmonization["dataset"] == dataset_name) &
            (df_harmonization["participant_id"] == row["participant_id"]) &
            (df_harmonization["session"] == row["session"])
        )

        # Update all present columns (both totals and individual items)
        for col in present_cols:
            df_harmonization.loc[mask, col] = row[col]

    return df_harmonization




###################################################################
#######################CLINICAL HARMONIZATION #####################
###################################################################

storage_root = r"../Storage Repository"
df_harmonization = pd.read_csv("modalities_info.csv")
df_participant_info = pd.read_csv("participants_info.csv")
df_participant_info.drop(columns="dataset", inplace=True)

df_harmonization = df_harmonization.merge(df_participant_info, on="participant_id", how="left")

Saved 9456 rows to modalities_info.csv

Saved combined CSV with 8563 rows across 34 datasets → participants_info.csv


In [17]:
df_modalities

,dataset,participant_id,session,anat,fmri,dwi,directory
0,ACHRP,ACHRPM80301092,A,1,1,0,E:\King's College London\Department of Psychos...
1,ACHRP,ACHRPM80303045,A,1,1,0,E:\King's College London\Department of Psychos...
2,ACHRP,ACHRPM80303730,A,1,1,0,E:\King's College London\Department of Psychos...
3,ACHRP,ACHRPM80304750,A,1,1,0,E:\King's College London\Department of Psychos...
4,ACHRP,ACHRPM80305418,A,1,1,0,E:\King's College London\Department of Psychos...
...,...,...,...,...,...,...,...
9451,YNC,YNCpc0218,01,1,1,0,E:\King's College London\Department of Psychos...
9452,YNC,YNCvd0034,01,1,1,0,E:\King's College London\Department of Psychos...
9453,YNC,YNCvd0084,01,1,1,0,E:\King's College London\Department of Psychos...
9454,YNC,YNCvd0115,01,1,1,0,E:\King's College London\Department of Psychos...


In [18]:
import os
import pandas as pd
import numpy as np

###########   MODALITY READING  ###################################
import warnings; warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
# Root folder containing all datasets
storage_root = r"../Storage Repository"

rows = []

# Loop through datasets (e.g., ACHRP, COBRE, MCIC)
for dataset in os.listdir(storage_root):
    dataset_path = os.path.join(storage_root, dataset)
    data_path = os.path.join(dataset_path, "data")

    # Skip if no data/ folder
    if not os.path.isdir(data_path):
        continue

    # Loop through subjects (sub-*)
    for sub in os.listdir(data_path):
        sub_path = os.path.join(data_path, sub)
        if not (os.path.isdir(sub_path) and sub.startswith("sub-")):
            continue

        # Check sessions (ses-*)
        session_folders = [s for s in os.listdir(sub_path) if os.path.isdir(os.path.join(sub_path, s)) and s.startswith("ses-")]

        # If no sessions, still check directly under subject
        if not session_folders:
            session_folders = [None]

        for ses in session_folders:
            ses_path = os.path.join(sub_path, ses) if ses else sub_path

            # Modality folders
            anat_path = os.path.join(ses_path, "anat")
            func_path = os.path.join(ses_path, "func")
            dwi_path  = os.path.join(ses_path, "dwi")

            # Existence flags (1 if folder exists and has ≥1 file)
            def has_files(path):
                return os.path.isdir(path) and any(os.path.isfile(os.path.join(path, f)) for f in os.listdir(path))

            anat_flag = int(has_files(anat_path))
            func_flag = int(has_files(func_path))
            dwi_flag  = int(has_files(dwi_path))

            session_name = ses.replace("ses-", "") if ses else "NA"

            rows.append({
                "dataset": dataset,
                "participant_id": sub.replace("sub-", ""),
                "session": session_name,
                "anat": anat_flag,
                "fmri": func_flag,
                "dwi": dwi_flag,
                "directory": os.path.abspath(ses_path)  
            })

# Build DataFrame and save combined CSV
df_modalities = pd.DataFrame(rows)
df_modalities.to_csv("modalities_info.csv", index=False)

print(f"Saved {len(df_modalities)} rows to modalities_info.csv")


###########   PARTICIPANT READING  ################################

all_dfs = []

# Loop through datasets (e.g., ACHRP, COBRE, MCIC)
for dataset in os.listdir(storage_root):
    dataset_path = os.path.join(storage_root, dataset)
    participants_file = os.path.join(dataset_path, "participants.csv")

    if os.path.isfile(participants_file):
        try:
            df = pd.read_csv(participants_file)

            # Remove unnamed columns
            df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

            # Add dataset name
            df["dataset"] = dataset

            all_dfs.append(df)

        except Exception as e:
            print(f"❌ Error reading {participants_file}: {e}")

# Concatenate all DataFrames
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)

    combined_df.to_csv("participants_info.csv", index=False)

    print(
        f"\nSaved combined CSV with {len(combined_df)} rows "
        f"across {combined_df['dataset'].nunique()} datasets "
        f"→ participants_info.csv"
    )
else:
    print("⚠️ No participants.csv files found.")




###################################################################




def merge_clinical_data1(df_harmonization, df_source, dataset_name, column_mapping):
    """
    Merge clinical variables (totals and individual items) from df_source into df_harmonization.

    Parameters:
    - df_harmonization: main dataframe (participants + modalities)
    - df_source: dataframe with clinical scores to merge
    - dataset_name: name of the dataset (string)
    - column_mapping: dict mapping source columns -> harmonized columns (e.g., items and totals)
    """

    # Rename source columns to harmonized names
    df_source = df_source.rename(columns=column_mapping)

    # Determine which columns are present in df_source
    present_cols = [col for col in df_source.columns if col in column_mapping.values()]

    # Loop over source rows and update harmonization dataframe
    for idx, row in df_source.iterrows():
        mask = (
            (df_harmonization["dataset"] == dataset_name) &
            (df_harmonization["participant_id"] == row["participant_id"]) &
            (df_harmonization["session"] == row["session"])
        )

        # Update all present columns (both totals and individual items)
        for col in present_cols:
            df_harmonization.loc[mask, col] = row[col]

    return df_harmonization




###################################################################
#######################CLINICAL HARMONIZATION #####################
###################################################################

storage_root = r"../Storage Repository"
df_harmonization = pd.read_csv("modalities_info.csv")
df_participant_info = pd.read_csv("participants_info.csv")
df_participant_info.drop(columns="dataset", inplace=True)

df_harmonization = df_harmonization.merge(df_participant_info, on="participant_id", how="left")

datasets = ["ACHRP", "AGERISK", "AOMICID", "AOMICPIOP", "AOMICPIOPII", "ARTS", "BCSPS", "BEACON", "BGSCHZ", "CANDI", "CARDS", "CATD", 
            "CBTYAD", "CMDPHC", "COBRE",
            "COMSS", "EDBPDTI", "MCIC", "MEGMMN", "MOA", "MUNI", "NBVSC", "NDARINV", "NUSDAST", "PAD", "PENNLEAD", "RSDHC", "SRPBS", 
            "SUDMEX", "TRFEP", "UCLACNP", "WMHSI", "YHA", "YNC"]



####################### PANSS #####################################
panss_columns = [
    "panss_p1","panss_p2","panss_p3","panss_p4","panss_p5","panss_p6","panss_p7",
    "panss_n1","panss_n2","panss_n3","panss_n4","panss_n5","panss_n6","panss_n7",
    "panss_g1","panss_g2","panss_g3","panss_g4","panss_g5","panss_g6","panss_g7",
    "panss_g8","panss_g9","panss_g10","panss_g11","panss_g12","panss_g13","panss_g14",
    "panss_g15","panss_g16", "panss_pos_total", "panss_neg_total", "panss_gen_total", "panss_total"]


positive_items = ["panss_p1","panss_p2","panss_p3","panss_p4","panss_p5","panss_p6","panss_p7"]
negative_items = ["panss_n1","panss_n2","panss_n3","panss_n4","panss_n5","panss_n6","panss_n7"]
general_items = ["panss_g1","panss_g2","panss_g3","panss_g4","panss_g5","panss_g6","panss_g7",
                 "panss_g8","panss_g9","panss_g10","panss_g11","panss_g12","panss_g13","panss_g14",
                 "panss_g15","panss_g16"]

# Add PANSS columns for all participants, filled with NaN
for col in panss_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan


for dataset in datasets:

    if dataset == "ARTS":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")
        #Add session
        df1["session"] = "A"
        
        # Define mapping for totals
        mapping = {
            "panss_pos": "panss_pos_total",
            "panss_neg": "panss_neg_total",
            "panss_gen": "panss_gen_total",
            "panss_tot": "panss_total"
        }
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)
        

    elif dataset == "BGSCHZ":
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/2079_PANSS_20250402.csv", skiprows=1)
        
        # Original session seems to just say baseline. 
        df1["session"] = "A"

        #Adapt ID from file
        df1["ID"] = dataset+df1["ID"]

        #replace MD with np.nan
        df1.replace("MD", np.nan, inplace=True)

        mapping = {
            f"FIPAN_{i}": f"{col}"
            for i, col in enumerate([
                # Positive items
                "panss_p1","panss_p2","panss_p3","panss_p4","panss_p5","panss_p6","panss_p7",
                # Negative items
                "panss_n1","panss_n2","panss_n3","panss_n4","panss_n5","panss_n6","panss_n7",
                # General items (16)
                "panss_g1","panss_g2","panss_g3","panss_g4","panss_g5","panss_g6","panss_g7",
                "panss_g8","panss_g9","panss_g10","panss_g11","panss_g12","panss_g13","panss_g14",
                "panss_g15","panss_g16"
            ], start=1)
        }

        # 👇 Add participant ID mapping after the comprehension
        mapping["ID"] = "participant_id"
        
        # Make VISIT categorical so 'b' comes first
        df1["VISIT"] = pd.Categorical(df1["VISIT"], categories=["b", "f", "other"], ordered=True)
        
        # Sort by ID, DAY_LAG ascending, and VISIT ('b' prioritized)
        df1.sort_values(by=["ID", "DAY_LAG", "VISIT"], inplace=True)
        
        # Keep only the first row per ID (drops other duplicates) in-place
        df1.drop_duplicates(subset="ID", keep="first", inplace=True)
        
        # Reset index if you want
        df1.reset_index(drop=True, inplace=True)
        
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

        #CALCULATE TOTALS
        # Filter df_harmonization to just the patients from this dataset
        mask = df_harmonization["dataset"] == dataset

        # Ensure all item columns are numeric
        all_items = positive_items + negative_items + general_items
        
        df_harmonization.loc[mask, all_items] = df_harmonization.loc[mask, all_items].apply(pd.to_numeric, errors='coerce')
        
        # Now calculate totals safely
        df_harmonization.loc[mask, "panss_pos_total"] = df_harmonization.loc[mask, positive_items].apply(
            lambda x: x.sum() if x.notna().any() else np.nan, axis=1
        )
        df_harmonization.loc[mask, "panss_neg_total"] = df_harmonization.loc[mask, negative_items].apply(
            lambda x: x.sum() if x.notna().any() else np.nan, axis=1
        )
        df_harmonization.loc[mask, "panss_gen_total"] = df_harmonization.loc[mask, general_items].apply(
            lambda x: x.sum() if x.notna().any() else np.nan, axis=1
        )
        df_harmonization.loc[mask, "panss_total"] = df_harmonization.loc[mask, all_items].apply(
            lambda x: x.sum() if x.notna().any() else np.nan, axis=1
        )


    elif dataset == "MEGMMN":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

        df1.columns = df1.columns.str.strip()

        #Add session
        df1["session"] = "0001"

        df1.replace("#N/A", np.nan, inplace=True)
        
        # Define mapping for totals
        mapping = {
            # Positive items
            "pos_p1": "panss_p1",
            "pos_p2": "panss_p2",
            "pos_p3": "panss_p3",
            "pos_p4": "panss_p4",
            "pos_p5": "panss_p5",
            "pos_p6": "panss_p6",
            "pos_p7": "panss_p7",
            
            # Negative items
            "neg_n1": "panss_n1",
            "neg_n2": "panss_n2",
            "neg_n3": "panss_n3",
            "neg_n4": "panss_n4",
            "neg_n5": "panss_n5",
            "neg_n6": "panss_n6",
            "neg_n7": "panss_n7",
            
            # General items
            "gps_g1": "panss_g1",
            "gps_g2": "panss_g2",
            "gps_g3": "panss_g3",
            "gps_g4": "panss_g4",
            "gps_g5": "panss_g5",
            "gps_g6": "panss_g6",
            "gps_g7": "panss_g7",
            "gps_g8": "panss_g8",
            "gps_g9": "panss_g9",
            "gps_g10": "panss_g10",
            "gps_g11": "panss_g11",
            "gps_g12": "panss_g12",
            "gps_g13": "panss_g13",
            "gps_g14": "panss_g14",
            "gps_g15": "panss_g15",
            "gps_g16": "panss_g16",
            
            # Totals
            "panss_positive": "panss_pos_total",
            "panss_negative": "panss_neg_total",
            "panss_general": "panss_gen_total",
            "panss_total": "panss_total"
        }
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)
        
    
    elif dataset == "NBVSC":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

        df1.columns = df1.columns.str.strip()

        #Add session
        df1["session"] = "A"

        # Define mapping for totals
        mapping = {
            "panns_total_sum": "panss_total"
        }
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


    elif dataset == "NDARINV":
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/phenotype/panss01.tsv", sep="\t")

        #Add session
        df1["session"] = "A"

        # Define mapping for totals
        mapping = {
            "pos_p1": "panss_p1",
            "pos_p2": "panss_p2",
            "pos_p3": "panss_p3",
            "pos_p4": "panss_p4",
            "pos_p5": "panss_p5",
            "pos_p6": "panss_p6",
            "pos_p7": "panss_p7",
            
            "neg_n1": "panss_n1",
            "neg_n2": "panss_n2",
            "neg_n3": "panss_n3",
            "neg_n4": "panss_n4",
            "neg_n5": "panss_n5",
            "neg_n6": "panss_n6",
            "neg_n7": "panss_n7",
            
            "gps_g1": "panss_g1",
            "gps_g2": "panss_g2",
            "gps_g3": "panss_g3",
            "gps_g4": "panss_g4",
            "gps_g5": "panss_g5",
            "gps_g6": "panss_g6",
            "gps_g7": "panss_g7",
            "gps_g8": "panss_g8",
            "gps_g9": "panss_g9",
            "gps_g10": "panss_g10",
            "gps_g11": "panss_g11",
            "gps_g12": "panss_g12",
            "gps_g13": "panss_g13",
            "gps_g14": "panss_g14",
            "gps_g15": "panss_g15",
            "gps_g16": "panss_g16",
            
            "panss_positive": "panss_pos_total",
            "panss_negative": "panss_neg_total",
            "panss_general": "panss_gen_total",
            "panss_total":"panss_total"
        }
        
        #Editions
        df1.rename(columns={"subjectkey":"participant_id"}, inplace = True)
        df1.replace(-999, np.nan, inplace=True)
        df1.replace(999, np.nan, inplace=True)

        df1["panss_total"] = df1[["panss_positive", "panss_negative", "panss_general"]].sum(axis=1, min_count=1)
        
        #Modify IDS
        df1["participant_id"] = df1["participant_id"].str.replace("_", "", regex=False)
        
        #Apply 
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

        
    
    elif dataset == "SRPBS":

        df1 =  pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/sup1.tsv", sep="\t")

        #replace MD with np.nan
        df1.replace("NA", np.nan, inplace=True)

        #Add session
        df1["session"] = "A"

        df1["participant_id"] = df1["participant_id"].str.replace("sub-", "SRPBS", regex=False)

        df1["panss_total"] = df1[["PANSS positive scale", "PANSS negative scale", "PANSS general psychopathology scale"]].sum(axis=1, min_count=1)
        
        mapping = {
                "PANSS positive scale": "panss_pos_total",   # PANSS positive scale
                "PANSS negative scale": "panss_neg_total",   # PANSS negative scale
                "PANSS general psychopathology scale":  "panss_gen_total",    # PANSS general psychopathology scale
                "panss_total": "panss_total"
            }

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)



        df2 =  pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/sup3.tsv", sep="\t")
        
        #replace MD with np.nan
        df2.replace("NA", np.nan, inplace=True)

        #Add session
        df2["session"] = "A"

        df2["participant_id"] = df2["participant_id"].str.replace("sub-", "SRPBS", regex=False)

        mapping = {
            # Totals
            "PANSS Positive Total": "panss_pos_total",
            "PANSS Negative Total": "panss_neg_total",
            "PANSS General Total":  "panss_gen_total",
            "PANSS Total (Positive + Negative + General)": "panss_total",
            
            # Positive items
            "PANSS Posi_1 Delusion": "panss_p1",
            "PANSS Posi_2 Conceptual disorganization": "panss_p2",
            "PANSS Posi_3 Hallucinatory behavior": "panss_p3",
            "PANSS Posi_4 Excitement": "panss_p4",
            "PANSS Posi_5 Grandiosity": "panss_p5",
            "PANSS Posi_6 Suspiciousness": "panss_p6",
            "PANSS Posi_7 Hostility": "panss_p7",
            
            # Negative items
            "PANSS Nega_1 Blunted affect": "panss_n1",
            "PANSS Nega_2 Emotional withdrawal": "panss_n2",
            "PANSS Nega_3 Poor rapport": "panss_n3",
            "PANSS Nega_4 Passive Apathetic social withdrawal": "panss_n4",
            "PANSS Nega_5 Difficulty in abstract thinking": "panss_n5",
            "PANSS Nega_6 Luck of spontaneity and flow of conversation": "panss_n6",
            "PANSS Nega_7 Stereotyped thinking": "panss_n7",
            
            # General psychopathology items
            "PANSS Gene_1 Somatic concern": "panss_g1",
            "PANSS Gene_2 Anxiety": "panss_g2",
            "PANSS Gene_3 Guilt feeling": "panss_g3",
            "PANSS Gene_4 Tension": "panss_g4",
            "PANSS Gene_5 Mannerisms and posturing": "panss_g5",
            "PANSS Gene_6 Depression": "panss_g6",
            "PANSS Gene_7 Motor retardation": "panss_g7",
            "PANSS Gene_8 Uncooperativeness": "panss_g8",
            "PANSS Gene_9 Unusual thought content": "panss_g9",
            "PANSS Gene_10 Disorientation": "panss_g10",
            "PANSS Gene_11 Poor attention": "panss_g11",
            "PANSS Gene_12 Lack of judgement and insight": "panss_g12",
            "PANSS Gene_13 Disturbance of volition": "panss_g13",
            "PANSS Gene_14 Poor impulse control": "panss_g14",
            "PANSS Gene_15 Preoccupation": "panss_g15",
            "PANSS Gene_16 Active social avoidance": "panss_g16"
        }

        df_harmonization = merge_clinical_data1(df_harmonization, df2, dataset, mapping)

        

        df3 =  pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/sup6.tsv", sep="\t")

        #replace MD with np.nan
        df3.replace("NA", np.nan, inplace=True)

        #Add session
        df3["session"] = "A"

        
        mapping = {
            "PANSS-positive": "panss_pos_total",
            "PANSS-negative": "panss_neg_total",
            "PANSS-general psychopathology scales": "panss_gen_total",
            "panss_total": "panss_total"
        }


        df3["participant_id"] = df3["participant_id"].str.replace("sub-", "SRPBS", regex=False)

        df3["panss_total"] = df3[["PANSS-positive", "PANSS-negative", "PANSS-general psychopathology scales"]].sum(axis=1, min_count=1)

        df_harmonization = merge_clinical_data1(df_harmonization, df3, dataset, mapping) 
        
    elif dataset == "COBRE":
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/1139_PANSS_20250402.csv", skiprows=1)
        
        # Original session seems to just say baseline. 
        df1["session"] = "A"
        
        #Adapt ID from file
        
        df1.rename(columns={"ID":"participant_id_old"}, inplace = True)
        
        #replace MD with np.nan
        df1.replace("MD", np.nan, inplace=True)
        
        #ID swap
        df_id_map = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/id_map.xlsx")
        
        df1 = df_id_map.merge(df1, on = "participant_id_old", how = "left")
        df1.drop(columns="participant_id_old", inplace = True)
        
        df1["participant_id"] = dataset + "00" + df1["participant_id"].astype(str)

        mapping = {
            f"FIPAN_{i}": f"{col}"
            for i, col in enumerate([
                # Positive items
                "panss_p1","panss_p2","panss_p3","panss_p4","panss_p5","panss_p6","panss_p7",
                # Negative items
                "panss_n1","panss_n2","panss_n3","panss_n4","panss_n5","panss_n6","panss_n7",
                # General items (16)
                "panss_g1","panss_g2","panss_g3","panss_g4","panss_g5","panss_g6","panss_g7",
                "panss_g8","panss_g9","panss_g10","panss_g11","panss_g12","panss_g13","panss_g14",
                "panss_g15","panss_g16"
            ], start=1)
        }

        # Calculate PANSS totals
        df1["panss_pos_total"] = df1[[f"FIPAN_{i}" for i in range(1, 8)]].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)
        df1["panss_neg_total"] = df1[[f"FIPAN_{i}" for i in range(8, 15)]].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)
        df1["panss_gen_total"] = df1[[f"FIPAN_{i}" for i in range(15, 31)]].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)
        df1["panss_total"] = df1[["panss_pos_total", "panss_neg_total", "panss_gen_total"]].sum(axis=1, min_count=1)

        mapping["panss_pos_total"] = "panss_pos_total"
        mapping["panss_neg_total"] = "panss_neg_total"
        mapping["panss_gen_total"] = "panss_gen_total"
        mapping["panss_total"] = "panss_total"
        
        # Make VISIT categorical so 'b' comes first
        df1["VISIT"] = pd.Categorical(df1["VISIT"], categories=["b", "f", "other"], ordered=True)
        
        # Sort by ID, DAY_LAG ascending, and VISIT ('b' prioritized)
        df1.sort_values(by=["participant_id", "DAY_LAG", "VISIT"], inplace=True)
        
        # Keep only the first row per ID (drops other duplicates) in-place
        df1.drop_duplicates(subset="participant_id", keep="first", inplace=True)
        
        # Reset index if you want
        df1.reset_index(drop=True, inplace=True)
        
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

    elif dataset == "MUNI":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

        df1.columns = df1.columns.str.strip()

        #Add session
        df1["session"] = "1"

        # Compute PANSS subscales
        panss_pos_items = ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]
        panss_neg_items = ["N1", "N2", "N3", "N4", "N5", "N6", "N7"]
        panss_gen_items = [
            "G1", "G2", "G3", "G4", "G5", "G6", "G7", "G8",
            "G9", "G10", "G11", "G12", "G13", "G14", "G15", "G16"
        ]
        
        # Safely convert to numeric (if some columns have strings or missing values)
        for col in panss_pos_items + panss_neg_items + panss_gen_items:
            df1[col] = pd.to_numeric(df1[col], errors="coerce")
        
        # Compute totals
        df1["panss_pos_total"] = df1[panss_pos_items].sum(axis=1, skipna=True, min_count=1)
        df1["panss_neg_total"] = df1[panss_neg_items].sum(axis=1, skipna=True, min_count=1)
        df1["panss_gen_total"] = df1[panss_gen_items].sum(axis=1, skipna=True, min_count=1)
        df1["panss_total"] = df1["panss_pos_total"] + df1["panss_neg_total"] + df1["panss_gen_total"]

        # Define mapping for totals
        mapping = {
            "P1": "panss_p1",
            "P2": "panss_p2",
            "P3": "panss_p3",
            "P4": "panss_p4",
            "P5": "panss_p5",
            "P6": "panss_p6",
            "P7": "panss_p7",
            "N1": "panss_n1",
            "N2": "panss_n2",
            "N3": "panss_n3",
            "N4": "panss_n4",
            "N5": "panss_n5",
            "N6": "panss_n6",
            "N7": "panss_n7",
            "G1": "panss_g1",
            "G2": "panss_g2",
            "G3": "panss_g3",
            "G4": "panss_g4",
            "G5": "panss_g5",
            "G6": "panss_g6",
            "G7": "panss_g7",
            "G8": "panss_g8",
            "G9": "panss_g9",
            "G10": "panss_g10",
            "G11": "panss_g11",
            "G12": "panss_g12",
            "G13": "panss_g13",
            "G14": "panss_g14",
            "G15": "panss_g15",
            "G16": "panss_g16",
            "panss_pos_total":"panss_pos_total",
            "panss_neg_total":"panss_neg_total",
            "panss_gen_total":"panss_gen_total",
            "panss_total":"panss_total"
        }

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)



###################################################################



####################### CDSS ######################################

cdss_columns = [
    "cdss_1",  # Depression
    "cdss_2",  # Hopelessness
    "cdss_3",  # Self-depreciation
    "cdss_4",  # Guilty ideas of reference
    "cdss_5",  # Pathological guilt
    "cdss_6",  # Morning depression
    "cdss_7",  # Early wakening
    "cdss_8",  # Suicide
    "cdss_9",  # Observed depression
    "cdss_total"
]

# Add PANSS columns for all participants, filled with NaN
for col in cdss_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan


for dataset in datasets:   
    if dataset == "BGSCHZ":
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/2079_Calgary_20250402.csv", skiprows=1)
        
        # Original session seems to just say baseline. 
        df1["session"] = "A"
        
        #Adapt ID from file
        df1["ID"] = dataset+df1["ID"]
        
        #replace MD with np.nan
        df1.replace("MD", np.nan, inplace=True)
        
        mapping = {
        f"FICAL_{i}": f"{col}"
        for i, col in enumerate(["cdss_1", "cdss_2", "cdss_3", "cdss_4", "cdss_5", "cdss_6", "cdss_7", "cdss_8", "cdss_9"], start=1)
        }
        
       
        df1[list(mapping.keys())] = df1[list(mapping.keys())].apply(pd.to_numeric, errors='coerce')
        df1['cdss_total'] = df1[['FICAL_1', 'FICAL_2', 'FICAL_3', 'FICAL_4', 'FICAL_5', 'FICAL_6', 'FICAL_7', 'FICAL_8', 'FICAL_9']].sum(axis=1, min_count=1)
        mapping['cdss_total'] = 'cdss_total'
        
        
        # Make VISIT categorical so 'b' comes first
        df1["VISIT"] = pd.Categorical(df1["VISIT"], categories=["b", "f", "other"], ordered=True)
        
        # Sort by ID, DAY_LAG ascending, and VISIT ('b' prioritized)
        df1.sort_values(by=["ID", "DAY_LAG", "VISIT"], inplace=True)
        
        # Keep only the first row per ID (drops other duplicates) in-place
        df1.drop_duplicates(subset="ID", keep="first", inplace=True)

        df1.rename(columns={"ID":"participant_id"}, inplace = True)
        
        # Reset index if you want
        df1.reset_index(drop=True, inplace=True)
        
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

    elif dataset == "MCIC":
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/5720_MCIC_Clinical_Data_20250415.csv", skiprows=1)
        
        mapping = {"MCICSHARE_35":"cdss_total"}

        # Original session seems to just say baseline. 
        df1["session"] = "A"
        
        #Adapt ID from file
        df1["ID"] = dataset+df1["ID"]
        
        #replace MD with np.nan
        df1.replace("DK", np.nan, inplace=True)
        df1.replace("MD", np.nan, inplace=True)

        #Order in case there is duplicates
        df1["VISIT"] = pd.Categorical(df1["VISIT"], categories=["BL", "f", "other"], ordered=True)

        # Sort by ID, DAY_LAG ascending, and VISIT ('b' prioritized)
        df1.sort_values(by=["ID", "DAY_LAG", "VISIT"], inplace=True)
        
        # Keep only the first row per ID (drops other duplicates) in-place
        df1.drop_duplicates(subset="ID", keep="first", inplace=True)

        df1.rename(columns={"ID":"participant_id"}, inplace = True)
        
        # Reset index if you want
        df1.reset_index(drop=True, inplace=True)

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)
        
    elif dataset == "NBVSC":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")
        #Add session
        df1["session"] = "A"
        
        # Define mapping for totals
        mapping = {
            "cdss_total": "cdss_total"
        }
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

    elif dataset == "COBRE":
        
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/1139_Calgary_20250402.csv", skiprows=1)
        
        # Original session seems to just say baseline. 
        df1["session"] = "A"
        
        #Adapt ID from file
        
        df1.rename(columns={"ID":"participant_id_old"}, inplace = True)
        
        #replace MD with np.nan
        df1.replace("MD", np.nan, inplace=True)
        
        #ID swap
        df_id_map = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/id_map.xlsx")
        
        df1 = df_id_map.merge(df1, on = "participant_id_old", how = "left")
        df1.drop(columns="participant_id_old", inplace = True)
        
        df1["participant_id"] = dataset + "00" + df1["participant_id"].astype(str)

        mapping = {
            f"FICAL_{i}": f"{col}"
            for i, col in enumerate([  
                "cdss_1",  # Depression
                "cdss_2",  # Hopelessness
                "cdss_3",  # Self-depreciation
                "cdss_4",  # Guilty ideas of reference
                "cdss_5",  # Pathological guilt
                "cdss_6",  # Morning depression
                "cdss_7",  # Early wakening
                "cdss_8",  # Suicide
                "cdss_9",  # Observed depression
            ])}

        # Calculate PANSS totals

        df1["cdss_total"] = df1[["FICAL_1", "FICAL_2", "FICAL_3", "FICAL_4", "FICAL_5", "FICAL_6", "FICAL_7", "FICAL_8", "FICAL_9"]].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)
        mapping["cdss_total"] = "cdss_total"
        
        # Make VISIT categorical so 'b' comes first
        df1["VISIT"] = pd.Categorical(df1["VISIT"], categories=["b", "f", "other"], ordered=True)
        
        # Sort by ID, DAY_LAG ascending, and VISIT ('b' prioritized)
        df1.sort_values(by=["participant_id", "DAY_LAG", "VISIT"], inplace=True)
        
        # Keep only the first row per ID (drops other duplicates) in-place
        df1.drop_duplicates(subset="participant_id", keep="first", inplace=True)
        
        # Reset index if you want
        df1.reset_index(drop=True, inplace=True)
        
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

###################################################################

        
####################### Hamilton Scale ############################


hamd_columns = [
    "hamilton_1",    # Depression
    "hamilton_2",    # Feelings of Guilt
    "hamilton_3",    # Suicide
    "hamilton_4",    # Insomnia, Early
    "hamilton_5",    # Insomnia, Middle
    "hamilton_6",    # Insomnia, Late
    "hamilton_7",    # Work Activities
    "hamilton_8",    # Retardation
    "hamilton_9",    # Agitation
    "hamilton_10",   # Anxiety, Psychic
    "hamilton_11",   # Anxiety, Somatic
    "hamilton_12",   # Somatic Symptoms, Gastrointestinal
    "hamilton_13",   # Somatic Symptoms, General
    "hamilton_14",   # Genital Symptoms
    "hamilton_15",   # Hypochondriasis
    "hamilton_16a",  # Loss of Weight: History
    "hamilton_16b",  # Loss of Weight: Actual Change
    "hamilton_17",   # Insight
    "hamilton_18a",  # Diurnal Variation: If No
    "hamilton_18b",  # Diurnal Variation: When Present
    "hamilton_19",   # Depersonalization and Derealization
    "hamilton_20",   # Paranoid Symptoms
    "hamilton_21",   # Obsessional and Compulsive Symptoms
    "hamilton_22",   # Fatigability
    "hamilton_23",   # Social Withdrawal
    "hamilton_24",   # Appetite Increase
    "hamilton_25",   # Increased Eating
    "hamilton_26",   # Carbohydrate Craving
    "hamilton_27",   # Weight Gain
    "hamilton_28",   # Hypersomnia
    "hamd_28", # HAMD Total 28 Score 
    "hamd_21", # HAMD Total 21 Score 
    "hamd_17", # HAMD Total 17 Score
    "hamd_6" # HAMD Total 6 Score: depressed mood (item 1), guilt (Item 2), work and activities (Item 7), retardation (Item 8), anxiety psychic (Item 10), and general somatic symptoms (Item 13)
]



# Add hamilton columns for all participants, filled with NaN
for col in hamd_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan


for dataset in datasets:
    if dataset == "UCLACNP":
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/hamilton.tsv", delimiter="\t")

        # Original session seems to just say baseline. 
        df1["session"] = "A"

        df1['participant_id'] = df1['participant_id'].str.replace(r'^sub-', 'UCLACNP', regex=True)
        
        #replace MD with np.nan
        df1.replace("n/a", np.nan, inplace=True)
        df1.replace(-9999, np.nan, inplace=True)

        mapping = {
        "hamilton1": "hamilton_1",      # Depression
        "hamilton2": "hamilton_2",      # Feelings of Guilt
        "hamilton3": "hamilton_3",      # Suicide
        "hamilton4": "hamilton_4",      # Insomnia, Early
        "hamilton5": "hamilton_5",      # Insomnia, Middle
        "hamilton6": "hamilton_6",      # Insomnia, Late
        "hamilton7": "hamilton_7",      # Work Activities
        "hamilton8": "hamilton_8",      # Retardation
        "hamilton9": "hamilton_9",      # Agitation
        "hamilton10": "hamilton_10",    # Anxiety, Psychic
        "hamilton11": "hamilton_11",    # Anxiety, Somatic
        "hamilton12": "hamilton_12",    # Somatic Symptoms, Gastrointestinal
        "hamilton13": "hamilton_13",    # Somatic Symptoms, General
        "hamilton14": "hamilton_14",    # Genital Symptoms
        "hamilton15": "hamilton_15",    # Hypochondriasis
        "hamilton16a": "hamilton_16a",  # Loss of Weight: History
        "hamilton16b": "hamilton_16b",  # Loss of Weight: Actual Change
        "hamilton17": "hamilton_17",    # Insight
        "hamilton18a": "hamilton_18a",  # Diurnal Variation: If No
        "hamilton18b": "hamilton_18b",  # Diurnal Variation: When Present
        "hamilton19": "hamilton_19",    # Depersonalization and Derealization
        "hamilton20": "hamilton_20",    # Paranoid Symptoms
        "hamilton21": "hamilton_21",    # Obsessional and Compulsive Symptoms
        "hamilton22": "hamilton_22",    # Fatigability
        "hamilton23": "hamilton_23",    # Social Withdrawal
        "hamilton24": "hamilton_24",    # Appetite Increase
        "hamilton25": "hamilton_25",    # Increased Eating
        "hamilton26": "hamilton_26",    # Carbohydrate Craving
        "hamilton27": "hamilton_27",    # Weight Gain
        "hamilton28": "hamilton_28",    # Hypersomnia
        "hamd_28": "hamd_28",           # HAMD Total 28 Score
        "hamd_21": "hamd_21",           # HAMD Total 21 Score
        "hamd_17": "hamd_17",            # HAMD Total 17 Score
        "hamd_6": "hamd_6"              # HAMD 6 items: 1, 2, 7, 8, 10, 13
        }

        # Calculate HAMD-6 total: items 1, 2, 7, 8, 10, 13
        hamd6_items = ["hamilton1", "hamilton2", "hamilton7", "hamilton8", "hamilton10", "hamilton13"]
        df1["hamd_6"] = df1[hamd6_items].sum(axis=1, min_count=1)
        
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


        
    elif dataset == "MOA":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/phenotype.xlsx")
        df1['participant_id'] = df1['participant_id'].str.replace(r'^sub-', '', regex=True)
        df1['session'] = df1['session'].str.replace(r'^ses-', '', regex=True)
        mapping = {"HAMD_Bech": "hamd_6", "HAM17_Total":"hamd_17"}
        df1.replace("n/a", np.nan, inplace=True)
        df1.replace(-9999, np.nan, inplace=True)
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


    elif dataset == "MUNI":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

        df1.columns = df1.columns.str.strip()

        #Add session
        df1["session"] = "1"

        #Calculate totals for MUNI
        # --- Calculate HAMD-6 total ---
        # HAMD-6 items: 1, 2, 7, 8, 10, 13
        hamd6_items = ["HMD1", "HMD2", "HMD7", "HMD8", "HMD10", "HMD13"]
        df1["hamd_6"] = df1[hamd6_items].sum(axis=1, min_count=1)
    
        # --- Calculate HAMD-17 total ---
        # Standard HAMD-17 items: 1-17
        hamd17_items = [f"HMD{i}" for i in range(1, 18)]
        df1["hamd_17"] = df1[hamd17_items].sum(axis=1, min_count=1)
    
        # --- Calculate HAMD-21 total ---
        # Standard HAMD-21 items: 1-17 + 18-21 (HMD18-HMD21)
        hamd21_items = [f"HMD{i}" for i in range(1, 22)]
        df1["hamd_21"] = df1[hamd21_items].sum(axis=1, min_count=1)
      
        # Define mapping for totals
        mapping = {
            "HMD1":  "hamilton_1",     # Depression
            "HMD2":  "hamilton_2",     # Feelings of Guilt
            "HMD3":  "hamilton_3",     # Suicide
            "HMD4":  "hamilton_4",     # Insomnia, Early
            "HMD5":  "hamilton_5",     # Insomnia, Middle
            "HMD6":  "hamilton_6",     # Insomnia, Late
            "HMD7":  "hamilton_7",     # Work Activities
            "HMD8":  "hamilton_8",     # Retardation
            "HMD9":  "hamilton_9",     # Agitation
            "HMD10": "hamilton_10",    # Anxiety, Psychic
            "HMD11": "hamilton_11",    # Anxiety, Somatic
            "HMD12": "hamilton_12",    # Somatic Symptoms, Gastrointestinal
            "HMD13": "hamilton_13",    # Somatic Symptoms, General
            "HMD14": "hamilton_14",    # Genital Symptoms
            "HMD15": "hamilton_15",    # Hypochondriasis
            "HMD16": "hamilton_16a",   # Loss of Weight
            "HMD17": "hamilton_17",    # Insight
            "HMD18": "hamilton_18a",   # Diurnal Variation
            "HMD19": "hamilton_19",    # Depersonalization and Derealization
            "HMD20": "hamilton_20",    # Paranoid Symptoms
            "HMD21": "hamilton_21",    # Obsessional and Compulsive Symptoms
            "HMD22": "hamilton_22",    # Fatigability
            "HMD23": "hamilton_23",    # Social Withdrawal
            "hamd_28": "hamd_28",           # HAMD Total 28 Score
            "hamd_21": "hamd_21",           # HAMD Total 21 Score
            "hamd_17": "hamd_17",            # HAMD Total 17 Score
            "hamd_6": "hamd_6"              # HAMD 6 items: 1, 2, 7, 8, 10, 13
        }
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


###################################################################



####################### MADRS #####################################

madrs_columns = [
    "madrs_1",   # Apparent sadness
    "madrs_2",   # Reported sadness
    "madrs_3",   # Inner tension
    "madrs_4",   # Reduced sleep
    "madrs_5",   # Reduced appetite
    "madrs_6",   # Concentration difficulties
    "madrs_7",   # Lassitude
    "madrs_8",   # Inability to feel
    "madrs_9",   # Pessimistic thoughts
    "madrs_10",  # Suicidal thoughts
    "madrs_total"  # MADRS Total Score (sum of 10 items)
]



# Add PANSS columns for all participants, filled with NaN
for col in madrs_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan


for dataset in datasets:
    if dataset == "NDARINV":

        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/phenotype/madrs01.tsv", sep="\t")

        #Add session
        df1["session"] = "A"

        mapping = {
            "madrssad": "madrs_2",      # Reported sadness
            "madrspes": "madrs_9",      # Pessimistic thoughts
            "madrssui": "madrs_10",     # Suicidal thoughts
            "madrsslp": "madrs_4",      # Reduced sleep
            "madrsfee": "madrs_8",      # Inability to feel
            "madrslas": "madrs_7",      # Lassitude
            "madrscon": "madrs_6",      # Concentration difficulties
            "madrsten": "madrs_3",      # Inner tension
            "madrsapp": "madrs_5",      # Reduced appetite
            "madrsaps": "madrs_1",      # Apparent sadness
            "madrstot": "madrs_total"   # MADRS total score
        }

        #Editions
        df1.rename(columns={"subjectkey":"participant_id"}, inplace = True)
        df1.replace(-999, np.nan, inplace=True)
        df1.replace(999, np.nan, inplace=True)
        
        #Modify IDS
        df1["participant_id"] = df1["participant_id"].str.replace("_", "", regex=False)
        
        #Apply 
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)
        
    elif dataset == "MOA":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/phenotype.xlsx")
        df1['participant_id'] = df1['participant_id'].str.replace(r'^sub-', '', regex=True)
        df1['session'] = df1['session'].str.replace(r'^ses-', '', regex=True)
        mapping = {"MADRS_Total": "madrs_total"}
        df1.replace("n/a", np.nan, inplace=True)
        df1.replace(-9999, np.nan, inplace=True)
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

    elif dataset == "RSDHC":

        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")
        #Add session
        df1["session"] = "A"

        df1.replace("n/a", np.nan, inplace=True)
        
        # Define mapping for totals
        mapping = {
            "MADRS": "madrs_total",
        }
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)
###################################################################



####################### BDI-II ####################################

bdi_columns = [
    "bdi_1",   # Sadness
    "bdi_2",   # Pessimism
    "bdi_3",   # Past failure
    "bdi_4",   # Loss of pleasure
    "bdi_5",   # Guilty feelings
    "bdi_6",   # Punishment feelings
    "bdi_7",   # Self-dislike
    "bdi_8",   # Self-criticalness
    "bdi_9",   # Suicidal thoughts or wishes
    "bdi_10",  # Crying
    "bdi_11",  # Agitation
    "bdi_12",  # Loss of interest
    "bdi_13",  # Indecisiveness
    "bdi_14",  # Worthlessness
    "bdi_15",  # Loss of energy
    "bdi_16",  # Changes in sleeping pattern
    "bdi_17",  # Irritability
    "bdi_18",  # Changes in appetite
    "bdi_19",  # Concentration difficulty
    "bdi_20",  # Tiredness or fatigue
    "bdi_21",  # Loss of interest in sex
    "bdi_total"  # BDI Total Score (sum of 21 items)
]

# Add PANSS columns for all participants, filled with NaN
for col in bdi_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan


for dataset in datasets:
    if dataset == "RSDHC":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")
        #Add session
        df1["session"] = "A"

        df1.replace("n/a", np.nan, inplace=True)
        
        # Define mapping for totals
        mapping = {
            "BDI": "bdi_total",
        }
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


        
    elif dataset == "SRPBS":
        ######sup1
        df1 =  pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/sup1.tsv", sep="\t")

        #replace MD with np.nan
        df1.replace("NA", np.nan, inplace=True)

        #Add session
        df1["session"] = "A"

        df1["participant_id"] = df1["participant_id"].str.replace("sub-", "SRPBS", regex=False)

        mapping = {"BDI-II": "bdi_total"}

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


        
        ######sup2
        df2 =  pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/sup2.tsv", sep="\t")
        
        #replace MD with np.nan
        df2.replace("NA", np.nan, inplace=True)

        #Add session
        df2["session"] = "A"

        df2["participant_id"] = df2["participant_id"].str.replace("sub-", "SRPBS", regex=False)

        mapping = {"BDI-II": "bdi_total"}

        df_harmonization = merge_clinical_data1(df_harmonization, df2, dataset, mapping)

        ######sup5
        df5 =  pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/sup5.tsv", sep="\t")
        
        #replace MD with np.nan
        df5.replace("NA", np.nan, inplace=True)

        #Add session
        df5["session"] = "A"

        df5["participant_id"] = df5["participant_id"].str.replace("sub-", "SRPBS", regex=False)

        mapping = {"BDI-II": "bdi_total"}

        df_harmonization = merge_clinical_data1(df_harmonization, df5, dataset, mapping)
        

        ######sup6
        df6 =  pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/sup6.tsv", sep="\t")
        
        #replace MD with np.nan
        df6.replace("NA", np.nan, inplace=True)

        #Add session
        df6["session"] = "A"

        df6["participant_id"] = df6["participant_id"].str.replace("sub-", "SRPBS", regex=False)

        mapping = {"BDI-II": "bdi_total"}

        df_harmonization = merge_clinical_data1(df_harmonization, df6, dataset, mapping)


        
        ######sup8
        df8 =  pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/sup8.tsv", sep="\t")
        
        #replace MD with np.nan
        df8.replace("NA", np.nan, inplace=True)

        #Add session
        df8["session"] = "A"

        df8["participant_id"] = df8["participant_id"].str.replace("sub-", "SRPBS", regex=False)

        mapping = {"BDI-II": "bdi_total"}

        df_harmonization = merge_clinical_data1(df_harmonization, df8, dataset, mapping)


        ######sup9
        df9 =  pd.read_csv(storage_root+"/"+dataset+"/"+"/tables/sup9.tsv", sep="\t")
        
        #replace MD with np.nan
        df9.replace("NA", np.nan, inplace=True)

        #Add session
        df9["session"] = "A"

        df9["participant_id"] = df9["participant_id"].str.replace("sub-", "SRPBS", regex=False)

        mapping = {"bdi": "bdi_total"}

        df_harmonization = merge_clinical_data1(df_harmonization, df9, dataset, mapping)

    elif dataset == "PENNLEAD":
        #IMPORTANT: this dataset has BDI child version, but not BDI-II adult. It might be good to incorporate. 
        continue


###################################################################


####################### SAPS ######################################

saps_columns = [
    "saps_1",   # Auditory Hallucinations
    "saps_2",   # Voices Commenting
    "saps_3",   # Voices Conversing
    "saps_4",   # Somatic or Tactile Hallucinations
    "saps_5",   # Olfactory Hallucinations
    "saps_6",   # Visual Hallucinations
    "saps_7",   # Global Rating of Hallucinations
    "saps_8",   # Persecutory Delusions
    "saps_9",   # Delusions of Jealousy
    "saps_10",  # Delusions of Guilt or Sin
    "saps_11",  # Grandiose Delusions
    "saps_12",  # Religious Delusions
    "saps_13",  # Somatic Delusions
    "saps_14",  # Delusions of Reference
    "saps_15",  # Delusions of Being Controlled
    "saps_16",  # Delusions of Mind Reading
    "saps_17",  # Thought Broadcasting
    "saps_18",  # Thought Insertion
    "saps_19",  # Thought Withdrawal
    "saps_20",  # Global Rating of Delusions
    "saps_21",  # Clothing and Appearance
    "saps_22",  # Social and Sexual Behavior
    "saps_23",  # Aggressive and Agitated Behavior
    "saps_24",  # Repetitive or Stereotyped Behavior
    "saps_25",  # Global Rating of Bizarre Behavior
    "saps_26",  # Derailment
    "saps_27",  # Tangentiality
    "saps_28",  # Incoherence
    "saps_29",  # Illogicality
    "saps_30",  # Circumstantiality
    "saps_31",  # Pressure of Speech
    "saps_32",  # Distractable Speech
    "saps_33",  # Changing
    "saps_34",  # Global Rating of Positive Formal Thought Disorder
    "saps_total"                     # SAPS Total Score (sum 1-34)
]

# Add PANSS columns for all participants, filled with NaN
for col in saps_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan


for dataset in datasets:

    if dataset == "UCLACNP":

        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/saps.tsv", delimiter="\t")

        # Original session seems to just say baseline. 
        df1["session"] = "A"
        df1['participant_id'] = df1['participant_id'].str.replace(r'^sub-', 'UCLACNP', regex=True)
        
        #replace MD with np.nan
        df1.replace("n/a", np.nan, inplace=True)
        df1.replace(-9999, np.nan, inplace=True)


        df1["saps_total"] = df1[[f"saps{i}" for i in range(1, 35)]].apply(pd.to_numeric, errors="coerce").sum(axis=1)

        mapping = {
            "saps1": "saps_1",           # Auditory Hallucinations
            "saps2": "saps_2",           # Voices Commenting
            "saps3": "saps_3",           # Voices Conversing
            "saps4": "saps_4",           # Somatic or Tactile Hallucinations
            "saps5": "saps_5",           # Olfactory Hallucinations
            "saps6": "saps_6",           # Visual Hallucinations
            "saps7": "saps_7",           # Global Rating of Hallucinations
            "saps8": "saps_8",           # Persecutory Delusions
            "saps9": "saps_9",           # Delusions of Jealousy
            "saps10": "saps_10",         # Delusions of Guilt or Sin
            "saps11": "saps_11",         # Grandiose Delusions
            "saps12": "saps_12",         # Religious Delusions
            "saps13": "saps_13",         # Somatic Delusions
            "saps14": "saps_14",         # Delusions of Reference
            "saps15": "saps_15",         # Delusions of Being Controlled
            "saps16": "saps_16",         # Delusions of Mind Reading
            "saps17": "saps_17",         # Thought Broadcasting
            "saps18": "saps_18",         # Thought Insertion
            "saps19": "saps_19",         # Thought Withdrawal
            "saps20": "saps_20",         # Global Rating of Delusions
            "saps21": "saps_21",         # Clothing and Appearance
            "saps22": "saps_22",         # Social and Sexual Behavior
            "saps23": "saps_23",         # Aggressive and Agitated Behavior
            "saps24": "saps_24",         # Repetitive or Stereotyped Behavior
            "saps25": "saps_25",         # Global Rating of Bizarre Behavior
            "saps26": "saps_26",         # Derailment
            "saps27": "saps_27",         # Tangentiality
            "saps28": "saps_28",         # Incoherence
            "saps29": "saps_29",         # Illogicality
            "saps30": "saps_30",         # Circumstantiality
            "saps31": "saps_31",         # Pressure of Speech
            "saps32": "saps_32",         # Distractable Speech
            "saps33": "saps_33",         # Clanging
            "saps34": "saps_34",         # Global Rating of Positive Formal Thought Disorder
            "saps_total":"saps_total"    # SAPS Total
        }
        
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


    
    elif dataset == "MCIC":
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/5720_MCIC_Clinical_Data_20250415.csv", skiprows=1)
        
        mapping = {
            "MCICSHARE_25": "saps_20",          # Global Rating of Severity of Delusions
            "MCICSHARE_26": "saps_7",     # Global Rating of Severity of Hallucinations
            "MCICSHARE_27": "saps_25",       # Global Rating of Severity of Bizarre Behavior
            "MCICSHARE_28": "saps_34",   # Global Rating of Positive Formal Thought Disorder
            }
        
        # Original session seems to just say baseline. 
        df1["session"] = "A"
        
        #Adapt ID from file
        df1["ID"] = dataset+df1["ID"]
        
        #replace MD with np.nan
        df1.replace("DK", np.nan, inplace=True)
        df1.replace("MD", np.nan, inplace=True)

        #Order in case there is duplicates
        df1["VISIT"] = pd.Categorical(df1["VISIT"], categories=["BL", "f", "other"], ordered=True)

        # Sort by ID, DAY_LAG ascending, and VISIT ('b' prioritized)
        df1.sort_values(by=["ID", "DAY_LAG", "VISIT"], inplace=True)
        
        # Keep only the first row per ID (drops other duplicates) in-place
        df1.drop_duplicates(subset="ID", keep="first", inplace=True)

        df1.rename(columns={"ID":"participant_id"}, inplace = True)
        
        # Reset index if you want
        df1.reset_index(drop=True, inplace=True)

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


    elif dataset == "WMHSI":

        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")
        #Add session
        df1["session"] = "A"
        
        # Define mapping for totals
        mapping = {
            "saps7": "saps_7",           # Global Rating of Hallucinations
            "saps20": "saps_20",         # Global Rating of Delusions
            "saps25": "saps_25",         # Global Rating of Bizarre Behavior
        }
        
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

    elif dataset == "NUSDAST":
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/SAPSANs/NUDataSharing_SAPSANs_n=571.csv")
     
        # Original session seems to just say baseline. 
        df1.rename(columns = {"Subject":"participant_id", "SAPSSANsSession": "session"}, inplace = True)

        df1['participant_id'] = dataset + df1['participant_id']
        df1["session"] = df1["session"].astype(str).str.slice(7)
        df1["session"] = df1["session"].str.replace('sapssans_0', 'A')
        
        
        #replace MD with np.nan
        df1.replace("U", np.nan, inplace=True)

        saps_cols = [f"saps{i}" for i in range(1, 35)]
        df1["saps_total"] = df1[saps_cols].apply(
            lambda row: pd.to_numeric(row, errors="coerce").sum() if not row.isna().all() else np.nan,
            axis=1
        )


        mapping = {
            "saps1": "saps_1",           # Auditory Hallucinations
            "saps2": "saps_2",           # Voices Commenting
            "saps3": "saps_3",           # Voices Conversing
            "saps4": "saps_4",           # Somatic or Tactile Hallucinations
            "saps5": "saps_5",           # Olfactory Hallucinations
            "saps6": "saps_6",           # Visual Hallucinations
            "saps7": "saps_7",           # Global Rating of Hallucinations
            "saps8": "saps_8",           # Persecutory Delusions
            "saps9": "saps_9",           # Delusions of Jealousy
            "saps10": "saps_10",         # Delusions of Guilt or Sin
            "saps11": "saps_11",         # Grandiose Delusions
            "saps12": "saps_12",         # Religious Delusions
            "saps13": "saps_13",         # Somatic Delusions
            "saps14": "saps_14",         # Delusions of Reference
            "saps15": "saps_15",         # Delusions of Being Controlled
            "saps16": "saps_16",         # Delusions of Mind Reading
            "saps17": "saps_17",         # Thought Broadcasting
            "saps18": "saps_18",         # Thought Insertion
            "saps19": "saps_19",         # Thought Withdrawal
            "saps20": "saps_20",         # Global Rating of Delusions
            "saps21": "saps_21",         # Clothing and Appearance
            "saps22": "saps_22",         # Social and Sexual Behavior
            "saps23": "saps_23",         # Aggressive and Agitated Behavior
            "saps24": "saps_24",         # Repetitive or Stereotyped Behavior
            "saps25": "saps_25",         # Global Rating of Bizarre Behavior
            "saps26": "saps_26",         # Derailment
            "saps27": "saps_27",         # Tangentiality
            "saps28": "saps_28",         # Incoherence
            "saps29": "saps_29",         # Illogicality
            "saps30": "saps_30",         # Circumstantiality
            "saps31": "saps_31",         # Pressure of Speech
            "saps32": "saps_32",         # Distractable Speech
            "saps33": "saps_33",         # Clanging
            "saps34": "saps_34",         # Global Rating of Positive Formal Thought Disorder
            "saps_total":"saps_total"    # SAPS Total
        }
 
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)



        
###################################################################


####################### SANS ######################################

sans_columns = [
    "sans_1",   # Unchanging Facial Expression
    "sans_2",   # Decreased Spontaneous Movements
    "sans_3",   # Paucity of Expressive Gestures
    "sans_4",   # Poor Eye Contact
    "sans_5",   # Affective Nonresponsivity
    "sans_6",   # Inappropriate Affect
    "sans_7",   # Lack of Vocal Inflections
    "sans_8",   # Global Rating of Affective Flattening
    "sans_9",   # Poverty of Speech
    "sans_10",  # Poverty of Content of Speech
    "sans_11",  # Blocking
    "sans_12",  # Increased Latency of Response
    "sans_13",  # Global Rating of Alogia
    "sans_14",  # Grooming and Hygiene
    "sans_15",  # Impersistence at Work or School
    "sans_16",  # Physical Anergia
    "sans_17",  # Global Rating of Avolition–Apathy
    "sans_18",  # Recreational Interests and Activities
    "sans_19",  # Sexual Interest and Activity
    "sans_20",  # Ability to Feel Intimacy and Closeness
    "sans_21",  # Relationships with Friends and Peers
    "sans_22",  # Global Rating of Anhedonia–Asociality
    "sans_23",  # Social Inactiveness
    "sans_24",  # Inattention During Mental Status Testing
    "sans_25",  # Global Rating of Attention
    "sans_total"  # SANS Total Score (sum 1–25)
]


# Add PANSS columns for all participants, filled with NaN
for col in sans_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan


for dataset in datasets:

    if dataset == "UCLACNP":

        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/sans.tsv", delimiter="\t")
        
        
        # Original session seems to just say baseline. 
        df1["session"] = "A"
        df1['participant_id'] = df1['participant_id'].str.replace(r'^sub-', 'UCLACNP', regex=True)
        
        #replace MD with np.nan
        df1.replace("n/a", np.nan, inplace=True)
        df1.replace(-9999, np.nan, inplace=True)

        
        df2 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/saps.tsv", delimiter="\t")[["participant_id", "saps35"]]
        # Original session seems to just say baseline. 
        df2['participant_id'] = df2['participant_id'].str.replace(r'^sub-', 'UCLACNP', regex=True)
        
        #replace MD with np.nan
        df2.replace("n/a", np.nan, inplace=True)
        df2.replace(-9999, np.nan, inplace=True)

        df1 = df1.merge(df2, on = "participant_id", how = "left")

        mapping = {
            "sans1": "sans_1",
            "sans2": "sans_2",
            "sans3": "sans_3",
            "sans4": "sans_4",
            "sans5": "sans_5",
            "saps35": "sans_6", # in SAPS
            "sans6": "sans_7",
            "sans7": "sans_8", 
            "sans8": "sans_9",
            #sans_10 Poverty of Content of Speech missing in UCLACNP
            "sans9": "sans_11",
            "sans10": "sans_12",
            "sans11": "sans_13",
            "sans12": "sans_14",
            "sans13": "sans_15",
            "sans15": "sans_16",
            "sans16": "sans_17",
            "sans17": "sans_18",
            "sans18": "sans_19", 
            "sans19": "sans_20",
            "sans20": "sans_21",
            "sans21": "sans_22",
            "sans22": "sans_23",
            "sans23": "sans_24",
            "sans24": "sans_25",
        }
         
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


    elif dataset == "MCIC":

        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/5720_MCIC_Clinical_Data_20250415.csv", skiprows=1)
        
        mapping = {
            "MCICSHARE_20": "sans_8",   # Global Rating of Affective Flattening
            "MCICSHARE_21": "sans_13",  # Global Rating of Alogia
            "MCICSHARE_22": "sans_17",  # Global Rating of Avolition - Apathy
            "MCICSHARE_23": "sans_22",  # Global Rating of Anhedonia - Asociality
            "MCICSHARE_24": "sans_25"   # Global Rating of Attention
        }

        # Original session seems to just say baseline. 
        df1["session"] = "A"
        
        #Adapt ID from file
        df1["ID"] = dataset+df1["ID"]
        
        #replace MD with np.nan
        df1.replace("DK", np.nan, inplace=True)
        df1.replace("MD", np.nan, inplace=True)

        #Order in case there is duplicates
        df1["VISIT"] = pd.Categorical(df1["VISIT"], categories=["BL", "f", "other"], ordered=True)

        # Sort by ID, DAY_LAG ascending, and VISIT ('b' prioritized)
        df1.sort_values(by=["ID", "DAY_LAG", "VISIT"], inplace=True)
        
        # Keep only the first row per ID (drops other duplicates) in-place
        df1.drop_duplicates(subset="ID", keep="first", inplace=True)

        df1.rename(columns={"ID":"participant_id"}, inplace = True)
        
        # Reset index if you want
        df1.reset_index(drop=True, inplace=True)

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)
       

    elif dataset == "WMHSI":

        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")
        #Add session
        df1["session"] = "A"
        
        # Define mapping for totals
        mapping = {
            "sans8": "sans_8",   # Global Rating of Affective Flattening
            "sans13": "sans_13",  # Global Rating of Alogia
            "sans17": "sans_17",  # Global Rating of Avolition - Apathy
            "sans22": "sans_22",  # Global Rating of Anhedonia - Asociality
            "sans25": "sans_25"   # Global Rating of Attention
        }
        
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

 
    elif dataset == "NUSDAST":
        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/SAPSANs/NUDataSharing_SAPSANs_n=571.csv")
     
        # Original session seems to just say baseline. 
        df1.rename(columns = {"Subject":"participant_id", "SAPSSANsSession": "session"}, inplace = True)

        df1['participant_id'] = dataset + df1['participant_id']
        df1["session"] = df1["session"].astype(str).str.slice(7)
        df1["session"] = df1["session"].str.replace('sapssans_0', 'A')
        
        #replace MD with np.nan
        df1.replace("U", np.nan, inplace=True)

        sans_cols = [f"sans{i}" for i in range(1, 26)]
        df1["sans_total"] = df1[sans_cols].apply(
            lambda row: pd.to_numeric(row, errors="coerce").sum() if not row.isna().all() else np.nan,
            axis=1
        )

        mapping = {
            "sans1": "sans_1",       
            "sans2": "sans_2",          
            "sans3": "sans_3",        
            "sans4": "sans_4",        
            "sans5": "sans_5",         
            "sans6": "sans_6",          
            "sans7": "sans_7",         
            "sans8": "sans_8",          
            "sans9": "sans_9",          
            "sans10": "sans_10",       
            "sans11": "sans_11",       
            "sans12": "sans_12",       
            "sans13": "sans_13",         
            "sans14": "sans_14",        
            "sans15": "sans_15",        
            "sans16": "sans_16",       
            "sans17": "sans_17",       
            "sans18": "sans_18",       
            "sans19": "sans_19",        
            "sans20": "sans_20",       
            "sans21": "sans_21",         
            "sans22": "sans_22",        
            "sans23": "sans_23",        
            "sans24": "sans_24",        
            "sans25": "sans_25",        
            "sans_total":"sans_total"    
        }
 
        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

    elif dataset == "MUNI":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

        df1.columns = df1.columns.str.strip()

        #Add session
        df1["session"] = "1"

        # Calculate sans_total as sum of S1–S24
        sans_items = [f"S{i}" for i in range(1, 25)]
        df1["sans_total"] = df1[sans_items].apply(lambda row: pd.to_numeric(row, errors="coerce").sum() 
                                                  if not row.isna().all() else np.nan, axis=1)

        # Define mapping for totals
        mapping = {
            "S1": "sans_1",
            "S2": "sans_2",
            "S3": "sans_3",
            "S4": "sans_4",
            "S5": "sans_5",
            "S6": "sans_6",
            "S7": "sans_7",
            "S8": "sans_8",
            "S9": "sans_9",
            "S10": "sans_10",
            "S11": "sans_11",
            "S12": "sans_12",
            "S13": "sans_13",
            "S14": "sans_14",
            "S15": "sans_15",
            "S16": "sans_16",
            "S17": "sans_17",
            "S18": "sans_18",
            "S19": "sans_19",
            "S20": "sans_20",
            "S21": "sans_21",
            "S22": "sans_22",
            "S23": "sans_23",
            "S24": "sans_24",
            "sans_total": "sans_total"
        }

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)
               
###################################################################


####################### BPRS ######################################


bprs_columns = [
    "bprs_1",   # Somatic Concern
    "bprs_2",   # Anxiety
    "bprs_3",   # Emotional Withdrawal
    "bprs_4",   # Conceptual Disorganization
    "bprs_5",   # Guilt Feelings
    "bprs_6",   # Tension
    "bprs_7",   # Mannerisms and Posturing
    "bprs_8",   # Grandiosity
    "bprs_9",   # Depressive Mood
    "bprs_10",  # Hostility
    "bprs_11",  # Suspiciousness
    "bprs_12",  # Hallucinatory Behavior
    "bprs_13",  # Motor Retardation
    "bprs_14",  # Uncooperativeness
    "bprs_15",  # Unusual Thought Content
    "bprs_16",  # Blunted Affect
    "bprs_17",  # Excitement
    "bprs_18",  # Disorientation
    "bprs_total"  # BPRS Total Score (sum of items 1–18)
]

# Add PANSS columns for all participants, filled with NaN
for col in bprs_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan


for dataset in datasets:

    if dataset in ["ACHRP", "AGERISK", "AOMICID", "AOMICPIOP", "AOMICPIOPII", "ARTS", "BCSPS", "BGSCHZ", "CANDI", "CARDS", "CMDPHC", 
            "COBRE", "COMSS", "EDBPDTI", "MCIC", "MEGMMN", "MOA", "NBVSC", "NDARINV", "NUSDAST", "PAD", "RSDHC", "SRPBS", 
            "SUDMEX", "TRFEP", "WMHSI", "YHA"]:
         continue


    elif dataset == "UCLACNP":

        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/bprs.tsv", delimiter="\t")
        
        # Original session seems to just say baseline. 
        df1["session"] = "A"
        df1['participant_id'] = df1['participant_id'].str.replace(r'^sub-', 'UCLACNP', regex=True)
        
        #replace MD with np.nan
        df1.replace("n/a", np.nan, inplace=True)
        df1.replace(-9999, np.nan, inplace=True)

        
        df1["bprs_total"] = df1[["bprs1",  "bprs2",  "bprs17", "bprs15", "bprs5",  "bprs19", "bprs24",  "bprs8", 
            "bprs3", "bprs6", "bprs9",  "bprs10", "bprs18", "bprs20",  
            "bprs11", "bprs16", "bprs21", "bprs14"]].apply(lambda row: pd.to_numeric(row, errors="coerce").sum() if not row.isna().all() else np.nan, axis=1)
                 

        mapping = {
            # 1: Somatic Concern
            "bprs1": "bprs_1",
            
            # 2: Anxiety
            "bprs2": "bprs_2",
        
            # 3: Emotional withdrawal
            "bprs17": "bprs_3",

            # 4: Conceptual Disorganization
            "bprs15": "bprs_4",
        
            # 5: Guilt Feelings
            "bprs5": "bprs_5",
        
            # 6: Tension
            "bprs19": "bprs_6",
        
            # 7: Mannerisms and posturing
            "bprs24": "bprs_7",
        
            # 8: Grandiosity
            "bprs8": "bprs_8",
        
            # 9: Depressive mood
            "bprs3": "bprs_9",
        
            # 10: Hostility
            "bprs6": "bprs_10",
        
            # 11: Suspiciousness
            "bprs9": "bprs_11",
        
            # 12: Hallucinatory behaviour
            "bprs10": "bprs_12",  
        
            # 13: Motor retardation
            "bprs18": "bprs_13",
        
            # 14: Uncooperativeness
            "bprs20": "bprs_14",
        
            # 15: Unusual thought content
            "bprs11": "bprs_15",
            
            # 16: Blunted Affect
            "bprs16": "bprs_16",
         
            # 19: Excitement
            "bprs21": "bprs_17",
        
            # 20: Disorientation
            "bprs14": "bprs_18",  

            # Total
            "bprs_total": "bprs_total"
        
        }

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

    elif dataset == "MUNI":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

        df1.columns = df1.columns.str.strip()

        #Add session
        df1["session"] = "1"


        # Define mapping for totals
        mapping = {
            "bprs": "bprs_total"
        }

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

bprs_columns = [
    "bprs_1",   # Somatic Concern
    "bprs_2",   # Anxiety
    "bprs_3",   # Emotional Withdrawal
    "bprs_4",   # Conceptual Disorganization
    "bprs_5",   # Guilt Feelings
    "bprs_6",   # Tension
    "bprs_7",   # Mannerisms and Posturing
    "bprs_8",   # Grandiosity
    "bprs_9",   # Depressive Mood
    "bprs_10",  # Hostility
    "bprs_11",  # Suspiciousness
    "bprs_12",  # Hallucinatory Behavior
    "bprs_13",  # Motor Retardation
    "bprs_14",  # Uncooperativeness
    "bprs_15",  # Unusual Thought Content
    "bprs_16",  # Blunted Affect
    "bprs_17",  # Excitement
    "bprs_18",  # Disorientation
    "bprs_total"  # BPRS Total Score (sum of items 1–18)
]

# Add PANSS columns for all participants, filled with NaN
for col in bprs_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan


for dataset in datasets:

    if dataset == "UCLACNP":

        df1 = pd.read_csv(storage_root+"/"+dataset+"/"+"tables/bprs.tsv", delimiter="\t")
        
        # Original session seems to just say baseline. 
        df1["session"] = "A"
        df1['participant_id'] = df1['participant_id'].str.replace(r'^sub-', 'UCLACNP', regex=True)
        
        #replace MD with np.nan
        df1.replace("n/a", np.nan, inplace=True)
        df1.replace(-9999, np.nan, inplace=True)

        
        df1["bprs_total"] = df1[["bprs1",  "bprs2",  "bprs17", "bprs15", "bprs5",  "bprs19", "bprs24",  "bprs8", 
            "bprs3", "bprs6", "bprs9",  "bprs10", "bprs18", "bprs20",  
            "bprs11", "bprs16", "bprs21", "bprs14"]].apply(lambda row: pd.to_numeric(row, errors="coerce").sum() if not row.isna().all() else np.nan, axis=1)
                 

        mapping = {
            # 1: Somatic Concern
            "bprs1": "bprs_1",
            
            # 2: Anxiety
            "bprs2": "bprs_2",
        
            # 3: Emotional withdrawal
            "bprs17": "bprs_3",

            # 4: Conceptual Disorganization
            "bprs15": "bprs_4",
        
            # 5: Guilt Feelings
            "bprs5": "bprs_5",
        
            # 6: Tension
            "bprs19": "bprs_6",
        
            # 7: Mannerisms and posturing
            "bprs24": "bprs_7",
        
            # 8: Grandiosity
            "bprs8": "bprs_8",
        
            # 9: Depressive mood
            "bprs3": "bprs_9",
        
            # 10: Hostility
            "bprs6": "bprs_10",
        
            # 11: Suspiciousness
            "bprs9": "bprs_11",
        
            # 12: Hallucinatory behaviour
            "bprs10": "bprs_12",  
        
            # 13: Motor retardation
            "bprs18": "bprs_13",
        
            # 14: Uncooperativeness
            "bprs20": "bprs_14",
        
            # 15: Unusual thought content
            "bprs11": "bprs_15",
            
            # 16: Blunted Affect
            "bprs16": "bprs_16",
         
            # 19: Excitement
            "bprs21": "bprs_17",
        
            # 20: Disorientation
            "bprs14": "bprs_18",  

            # Total
            "bprs_total": "bprs_total"
        
        }

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

    elif dataset == "MUNI":
        df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

        df1.columns = df1.columns.str.strip()

        #Add session
        df1["session"] = "1"


        # Define mapping for totals
        mapping = {
            "bprs": "bprs_total"
        }

        df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


######################################################################################################################################
######################################################################################################################################
######################################################################################################################################
######################################################  FRAN   #######################################################################
######################################################################################################################################
######################################################################################################################################

#####################
#####################
###### MEGMMN #######
#####################
#####################

dataset = "MEGMMN"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

df1.columns = df1.columns.str.strip()

##############################################
##############################################
############### GF SCALE #####################
##############################################
##############################################


#### OFFICIAL COLUMN LIST FOR SCALE HERE #####

standard_global_functioning = ["gf_role_score", "gf_role_low", "gf_role_high", "gf_social_score", "gf_social_low", "gf_social_high"]

#### DATASET COLUMN LIST FOR SCALE HERE ######

scale_global_functioning = ["gf_role_scole", "gf_role_low", "gf_role_high", "gf_social_scale", "gf_social_low", "gf_social_high"]

for col in standard_global_functioning:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

#Add session
df1["session"] = "0001"
df1.replace("#N/A", np.nan, inplace=True)

# Define mapping for totals
mapping = {
    "gf_role_scole": "gf_role_score",
    "gf_role_low": "gf_role_low",
    "gf_role_high": "gf_role_high",
    "gf_social_scale": "gf_social_score",
    "gf_social_low": "gf_social_low",
    "gf_social_high": "gf_social_high"
}


df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)
df_harmonization.to_csv("metafile.csv", index=False)


####################################################
####################################################
################### COGNITIVE & IQ #################
####################################################
####################################################


#### OFFICIAL COLUMN LIST FOR SCALE HERE #####

MEGMMN_columns = [
    "untreated_psychosis_days",
    "medication_days_to_scan",
    "handedness", # 0 is Right and 1 is Left
    "hollingshead_4f_socio_economic_status",
    "parents_hollingshead_4f_socio_economic_status",
    "attention_vigilance_percentile",
    "attention_vigilance_t_score",
    "bacs_symbol_coding_percentile",
    "bacs_symbol_coding_raw",
    "bacs_symbol_coding_t_score",
    "bvmt_r_percentile",
    "bvmt_r_raw_sum",
    "bvmt_r_t_score",
    "bvmt_r_trial_1_raw",
    "bvmt_r_trial_2_raw",
    "bvmt_r_trial_3_raw",
    "category_fluency_percentile",
    "category_fluency_raw",
    "category_fluency_t_score",
    "cpt_ip_2_digit_raw",
    "cpt_ip_3_digit_raw",
    "cpt_ip_4_digit_raw",
    "cpt_ip_mean_raw",
    "cpt_ip_percentile",
    "cpt_ip_t_score",
    "full_scale_iq_2subtest",
    "full_scale_iq_2subtest_percentile",
    "full_scale_iq_2subtest_t_score",
    "hvlt_r_percentile",
    "hvlt_r_raw_sum",
    "hvlt_r_t_score",
    "hvlt_r_trial_1_raw",
    "hvlt_r_trial_2_raw",
    "hvlt_r_trial_3_raw",
    "letter_number_span_percentile",
    "letter_number_span_raw",
    "letter_number_span_t_score",
    "matrix_reasoning_raw",
    "matrix_reasoning_t_score",
    "mccb_overall_percentile",
    "mccb_overall_t_score",
    "msceit_managing_emotions_percentile",
    "msceit_managing_emotions_raw",
    "msceit_managing_emotions_t_score",
    "nab_mazes_percentile",
    "nab_mazes_raw",
    "nab_mazes_t_score",
    "reasoning_problem_solving_percentile",
    "reasoning_problem_solving_t_score",
    "social_cognition_percentile",
    "social_cognition_t_score",
    "speed_of_processing_percentile",
    "speed_of_processing_t_score",
    "trail_making_test_a_percentile",
    "trail_making_test_a_raw",
    "trail_making_test_a_t_score",
    "verbal_learning_percentile",
    "verbal_learning_t_score",
    "visual_learning_percentile",
    "visual_learning_t_score",
    "vocabulary_raw",
    "vocabulary_t_score",
    "wms_iii_spatial_span_percentile",
    "wms_iii_spatial_span_raw",
    "wms_iii_spatial_span_t_score",
    "working_memory_percentile",
    "working_memory_t_score",
]

for col in MEGMMN_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

#Add session
df1["session"] = "0001"
df1.replace("#N/A", np.nan, inplace=True)

#### DATASET COLUMN LIST FOR SCALE HERE ######

cognitive_mapping = {
    "DUP": "untreated_psychosis_days",
    "MED2SCAN": "medication_days_to_scan",
    "HAND": "handedness",
    "SES": "self_hollingshead_4f_socio_economic_status",
    "PSES": "parents_hollingshead_4f_socio_economic_status",
    "ATT_VIGPCT": "attention_vigilance_percentile",
    "ATT_VIGTSCR": "attention_vigilance_t_score",
    "BACSSCPCT": "bacs_symbol_coding_percentile",
    "BACSSCRAW": "bacs_symbol_coding_raw",
    "BACSSCTSCR": "bacs_symbol_coding_t_score",
    "BVMTRPCT": "bvmt_r_percentile",
    "BVMTRRAWSUM": "bvmt_r_raw_sum",
    "BVMTRRAWT1": "bvmt_r_trial_1_raw",
    "BVMTRRAWT2": "bvmt_r_trial_2_raw",
    "BVMTRRAWT3": "bvmt_r_trial_3_raw",
    "BVMTRTSCR": "bvmt_r_t_score",
    "CPTIPPCT": "cpt_ip_percentile",
    "CPTIPRAW2D": "cpt_ip_2_digit_raw",
    "CPTIPRAW3D": "cpt_ip_3_digit_raw",
    "CPTIPRAW4D": "cpt_ip_4_digit_raw",
    "CPTIPRAWMEAN": "cpt_ip_mean_raw",
    "CPTIPTSCR": "cpt_ip_t_score",
    "FLUENPCT": "fluency_percentile",
    "FLUENRAW": "fluency_raw",
    "FLUENTSCR": "fluency_t_score",
    "FULL2IQ": "full_scale_iq_2subtest",
    "FULL2PCT": "full_scale_iq_2subtest_percentile",
    "FULL2TS": "full_scale_iq_2subtest_t_score",
    "HVLTRPCT": "hvlt_r_percentile",
    "HVLTRRAWSUM": "hvlt_r_raw_sum",
    "HVLTRRAWT1": "hvlt_r_trial_1_raw",
    "HVLTRRAWT2": "hvlt_r_trial_2_raw",
    "HVLTRRAWT3": "hvlt_r_trial_3_raw",
    "HVLTRTSCR": "hvlt_r_t_score",
    "LNSPCT": "letter_number_span_percentile",
    "LNSRAW": "letter_number_span_raw",
    "LNSTSCR": "letter_number_span_t_score",
    "MATRIX_RS": "matrix_reasoning_raw",
    "MATRIX_TS": "matrix_reasoning_t_score",
    "MSCEITMEPCT": "msceit_managing_emotions_percentile",
    "MSCEITMERAW": "msceit_managing_emotions_raw",
    "MSCEITMETSCR": "msceit_managing_emotions_t_score",
    "NABPCT": "nab_mazes_percentile",
    "NABRAW": "nab_mazes_raw",
    "NABTSCR": "nab_mazes_t_score",
    "OVERALLPCT": "mccb_overall_percentile",
    "OVERALLTSCR": "mccb_overall_t_score",
    "RPSPCT": "reasoning_problem_solving_percentile",
    "RPSTSCR": "reasoning_problem_solving_t_score",
    "SOCCOGPCT": "social_cognition_percentile",
    "SOCCOGTSCR": "social_cognition_t_score",
    "SPEEDPCT": "speed_of_processing_percentile",
    "SPEEDTSCR": "speed_of_processing_t_score",
    "TMTPCT": "trail_making_test_a_percentile",
    "TMTRAW": "trail_making_test_a_raw",
    "TMTTSCR": "trail_making_test_a_t_score",
    "VERBPCT": "verbal_learning_percentile",
    "VERBTSCR": "verbal_learning_t_score",
    "VISPCT": "visual_learning_percentile",
    "VISTSCR": "visual_learning_t_score",
    "VOCAB_RS": "vocabulary_raw",
    "VOCAB_TS": "vocabulary_t_score",
    "WMPCT": "working_memory_percentile",
    "WMTSCR": "working_memory_t_score",
    "WMS3SSPCT": "wms_iii_spatial_span_percentile",
    "WMS3SSRAW": "wms_iii_spatial_span_raw",
    "WMS3SSTSCR": "wms_iii_spatial_span_t_score",
}

df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)



#####################
#####################
####### MUNI ########
#####################
#####################



dataset = "MUNI"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

df1.columns = df1.columns.str.strip()

df1["session"] = "1"

standard_columns = [
    "handedness", "height", "weight", "BMI", "education_years", "transition", "transition_time_days", "transition_time_years",
    "on_medication", "age_at_onset", "illness_duration", "number_of_hospitals", "gaf_total", "alcohol_g/day", "cigars/day", 
    "bprs", "epsroh", "espgrad", "tdrs", "aims"]


#"STANDARDISE_WST_IQ", "STANDARDISE_ARMS", "STANDARDISE_Age_FU", "STANDARDISE_INTERVAL",
# "STANDARDISE_FU_GROUP_ARMS", "STANDARDISE_PTBS", "AGE_ONSET_COMP", "AGE_ONSET_CHECK", 
# "AP_YESNO", "AD_YESNO", "TOTAL_AD", "MOOD_YESNO", "TYPICAL_ATYPICAL_YESNO", "TYPICAL_AP_YESNO",
# "ATYPICAL_AP_YESNO", "TYPICAL_AP", "ATYPICAL_AP", "TOTAL_AP", "CPZ_EQ", "phillips"

for col in standard_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

MUNI_columns = [
    "HANDEDNESS", "HEIGHT", "WEIGHT", "BMI", "SCHOOL", "TRANSITION", "TRANSITION_TIME_DAYS", "TRANSITION_TIME_YEARS", 
    "MD_CURR", "AGE_ONSET", "ILLDUR_YEARS", "NUM_HOSP",
    "GAF_aufn", "alkgramm", "nikotin", "bprs", "epsroh",
    "epsgrad", "tdrs", "aims"]

mapping = {
    "HANDEDNESS": "handedness",
    "HEIGHT": "height",
    "WEIGHT": "weight",
    "BMI": "BMI",
    "SCHOOL": "education_years",
    "TRANSITION": "transition",
    "TRANSITION_TIME_DAYS": "transition_time_days",
    "TRANSITION_TIME_YEARS": "transition_time_years",
    "MD_CURR": "on_medication",
    "AGE_ONSET": "age_at_onset",
    "ILLDUR_YEARS": "illness_duration",
    "NUM_HOSP": "number_of_hospitals",
    "GAF_aufn": "gaf_total",
    "alkgramm": "alcohol_g/day",
    "nikotin": "cigars/day",
    "epsroh": "epsroh",
    "epsgrad": "espgrad",
    "tdrs": "tdrs",
    "aims": "aims"
}

df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


######################
######################
####### BCSPS ########
######################
######################


dataset = "BCSPS"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

df1.columns = df1.columns.str.strip()

df1["session"] = "A"

standard_columns = ["iq_tap", "psyrats_auditory"]

for col in standard_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

        
BCSPS_columns = ["iq", "psyrats"]

mapping = {
    "iq": "iq_tap",
    "psyrats": "psyrats_auditory"
}


df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


######################
######################
####### CANDI ########
######################
######################


dataset = "CANDI"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

df1.columns = df1.columns.str.strip()

df1["session"] = "A"

standard_columns = ["weight", "height", "head_circumference", "tanner_stage"]

for col in standard_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

        
CANDI_columns = ["Weight", "Height", "Head_Circumference (cm)", "Tanner_Stage"]

mapping = {
    "Weight": "weight",
    "Height": "height",
    "Head_Circumference (cm)": "head_circumference",
    "Tanner_Stage": "tanner_stage"
}


df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


###############################
###############################
####### CARDS tab_data ########
###############################
###############################


dataset = "CARDS"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

df1.columns = df1.columns.str.strip()

df1["session"] = "A"

standard_columns = ["ethnicity", "handedness"]

for col in standard_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

        
CARDS_columns = ["ethnicity", "handedness_std"]

mapping = {
    "Weight": "weight",
    "Height": "height",
    "Head_Circumference (cm)": "head_circumference",
    "Tanner_Stage": "tanner_stage"
}


df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

###############################
###############################
######### CARDS Rest ##########
###############################
###############################

dataset = "CARDS"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/CARDS_master_database - Copy.xlsx")

df1.columns = df1.columns.str.strip()

df1["session"] = "A"

standard_columns = ["bnss_total", "cgi_severity", "medication", "medication_dosage", "antipsychotic_generation",]

#cpz, 
for col in standard_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

        
CARDS_columns = ["bnss", "cgi", "medication", "dose", "generation"]

mapping = {
    "bnss": "bnss_total",
    "cgi": "cgi_severity",
    "medication": "medication",
    "dose": "medication_dosage",
    "generation": "antipsychotic_generation"
}


df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


###############################
###############################
############ CMDPHC ###########
###############################
###############################

dataset = "CMDPHC"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

df1.columns = df1.columns.str.strip()

df1["session"] = "01"

standard_columns = ["psychiatric_comorbidities", "medication_load", "iq_NART"]


for col in standard_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

        
CMDPHC_columns = ["number_comorbid_dx", "medload", "iq"]

mapping = {
    "number_comorbid_dx": "psychiatric_comorbidities",
    "medload": "medication_load",
    "iq": "iq_NART"
}


df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


###############################
###############################
############ COMSS ############
###############################
###############################

dataset = "COMSS"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

df1.columns = df1.columns.str.strip()

df1["session"] = "A"

standard_columns = ["smoking"] # 1 smoker 0 non


for col in standard_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

        
CMDPHC_columns = ["smoking_status"]

mapping = {
    "smoking_status": "smoking"
}


df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


###############################
###############################
############ RSDHC ############
###############################
###############################

dataset = "RSDHC"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/tab_data.xlsx")

df1.columns = df1.columns.str.strip()

df1["session"] = "A"

standard_columns = ["iq_RAVEN", "zung", "hads_anxiety", "hads_depression", "sds", "tas_26", "ecr_avoidant_attach", 
                    "ecr_anxious_attach", "rrs_total", "rrs_reflection", "rrs_brooding", "rrs_depression", "edinburgh"] 


for col in standard_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

        

RSDHC_columns = [
    "IQ_Raven",
    "Zung_SDS",
    "HADS-anx",
    "HADS-depr",
    "MC-SDS",
    "TAS-26",
    "ECR-avoid",
    "ECR-anx",
    "RRS-sum",
    "RRS-reflection",
    "RRS-brooding",
    "RRS-depr",
    "Edinburgh"
]



mapping = {
    "IQ_Raven": "iq_RAVEN",
    "Zung_SDS": "zung",
    "HADS-anx": "hads_anxiety",
    "HADS-depr": "hads_depression",
    "MC-SDS": "sds",
    "TAS-26": "tas_26",
    "ECR-avoid": "ecr_avoidant_attach",
    "ECR-anx": "ecr_anxious_attach",
    "RRS-sum": "rrs_total",
    "RRS-reflection": "rrs_reflection",
    "RRS-brooding": "rrs_brooding",
    "RRS-depr": "rrs_depression",
    "Edinburgh": "edinburgh"
}



df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)


###############################
###############################
############ SRPBS ############
###############################
###############################

dataset = "SRPBS"

df1 = pd.read_excel(storage_root+"/"+dataset+"/"+"tables/table_updated.xlsx")

df1.columns = df1.columns.str.strip()

df1["session"] = "A"


for col in standard_columns:
    if col not in df_harmonization.columns:
        df_harmonization[col] = np.nan

        
SRPBS_columns = [
    "hand",
    "iq_unknown", 
    "iq_wais_iii", 
    "iq_wais_iii_short",
    "FIQ",
    "VIQ",
    "PIQ",
    "BDI-II",
    "AQ(total)",
    "AQ(ss)",
    "AQ(as)",
    "AQ(atd)",
    "AQ(Com)",
    "AQ(imag)",
    "PDI yes/no score",
    "PDI distress score",
    "PDI preoccupation score",
    "PDI conviction score",
    "ADOSver",
    "ADOS-A",
    "ADOS-B",
    "Total",
    "Imagination",
    "RRB",
    "PANSS positive scale",
    "PANSS negative scale",
    "PANSS general psychopathology scale"
]

mapping = {
    "hand": "handedness",
    "iq_unknown": "estimated_iq_unknown_method",
    "iq_wais_iii": "estimated_iq_wais_iii",
    "iq_wais_iii_short": "estimated_iq_wais_iii_short",
    "FIQ": "wais_derived_full_scale_iq",
    "VIQ": "wais_derived_verbal_iq",
    "PIQ": "wais_derived_performance_iq",
    "AQ(total)": "aq_total", #autism quaocient
    "AQ(ss)": "aq_social_skills",
    "AQ(as)": "aq_attention_switching",
    "AQ(atd)": "aq_attention_to_detail",
    "AQ(Com)": "aq_communication",
    "AQ(imag)": "aq_imagination",
    "PDI yes/no score": "pdi_endorsed_beliefs_count", # Peters Delusions Inventory (PDI)
    "PDI distress score": "pdi_distress",
    "PDI preoccupation score": "pdi_preoccupation",
    "PDI conviction score": "pdi_conviction",
    "ADOSver": "ados_version", #ADOS (Autism Diagnostic Observation Schedule)
    "ADOS-A": "ados_social_affect",
    "ADOS-B": "ados_rrb",
    "Total": "ados_total",
    "Imagination": "ados_imagination"
}


df_harmonization = merge_clinical_data1(df_harmonization, df1, dataset, mapping)

###################################################################
###################################################################
###################################################################
###################################################################
###################################################################
###################################################################


df_harmonization.to_csv("metafile.csv", index=False)

Saved 9456 rows to modalities_info.csv

Saved combined CSV with 8563 rows across 34 datasets → participants_info.csv


In [19]:
import pandas as pd

# Path to your Excel file
file_path = r"C:\franc\sergi\King's College London\Department of Psychosis Shared Database Initiative - Documents\Storage Repository\MOA\tables\tab_data.xlsx"

# Load the Excel file
df = pd.read_excel(file_path, engine="openpyxl")  # engine helps avoid warnings

# Create a standardized binary column from 'smoking_status'
# Maps: 'nonsmoking' -> 0, 'smoking' -> 1; leaves anything else as NaN
df["medication"] = (
    df["smoking_status"]
      .astype(str)
      .str.strip()
      .str.lower()
      .map({
          "nonsmoking": 0,
          "smoking": 1
      })
)

# Save back to the same Excel file
df.to_excel(file_path, index=False, engine="openpyxl")

FileNotFoundError: [Errno 2] No such file or directory: "C:\\franc\\sergi\\King's College London\\Department of Psychosis Shared Database Initiative - Documents\\Storage Repository\\MOA\\tables\\tab_data.xlsx"

In [ ]:
import re
from pathlib import Path
import pandas as pd

# Path to your Excel file
file_path = r"C:\Users\franc\King's College London\Department of Psychosis Shared Database Initiative - Documents\Storage Repository\SRPBS\tables\test.xlsx"

# Load the Excel file
df = pd.read_excel(file_path, engine="openpyxl")

# --- Safety checks ---
required_cols = {"WAIS", "Estimated IQ"}
missing = required_cols - set(df.columns)
if missing:
    raise KeyError(f"Missing required column(s): {', '.join(missing)}")

# --- Normalization helper for WAIS values ---
def normalize_wais(val):
    """
    Normalize WAIS entries by:
    - Converting to string, lowercasing
    - Stripping whitespace
    - Removing punctuation (.,-/_)
    - Collapsing multiple spaces
    So variants like 'WAISIII short ver.', 'wais-iii short ver', ' NA ' normalize consistently.
    """
    if pd.isna(val):
        return "unknown"
    s = str(val).strip().lower()
    # common null-like
    if s in {"na", "n/a", "none", ""}:
        return "unknown"
    # remove punctuation and underscores/slashes/dashes/periods
    s = re.sub(r"[.\-_/]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    # remove spaces to simplify matching like "wais iii"
    s_nospace = s.replace(" ", "")
    return s_nospace

wais_norm = df["WAIS"].map(normalize_wais)


mapping = {
    "unknown": "unknown",
    "waisiii": "wais_iii",
    "waisiiishortver": "wais_iii_short",
}

canonical = wais_norm.map(mapping)

# --- Prepare destination columns ---
for col in ["iq_unknown", "iq_wais_iii", "iq_wais_iii_short"]:
    if col not in df.columns:
        df[col] = pd.NA

# Ensure Estimated IQ is numeric
est_iq = pd.to_numeric(df["Estimated IQ"], errors="coerce")

# Fill based on canonical category
df.loc[canonical.eq("unknown"), "iq_unknown"] = est_iq
df.loc[canonical.eq("wais_iii"), "iq_wais_iii"] = est_iq
df.loc[canonical.eq("wais_iii_short"), "iq_wais_iii_short"] = est_iq

# --- Optional: report unexpected values to help data cleaning ---
unexpected_mask = ~canonical.isin({"unknown", "wais_iii", "wais_iii_short"}) & ~canonical.isna()
unexpected_raw = df.loc[unexpected_mask, "WAIS"].astype(str).unique()
if len(unexpected_raw) > 0:
    print("Unrecognized WAIS values found (left unmapped):")
    for v in unexpected_raw:
        print(f"  - {v}")

# --- Save to table_updated.xlsx in the same directory ---
in_path = Path(file_path)
out_path = in_path.with_name("table_updated.xlsx")
df.to_excel(out_path, index=False, engine="openpyxl")
print(f"Saved updated table to: {out_path}")